# 📊 Statistical Analysis & Feature Engineering
**Egypt Real Estate Market Analyzer**

This notebook performs deep statistical analysis on our cleaned dataset. We will:
1. Engineer financial metrics like `price_per_sqft`.
2. Analyze distributions and correlations.
3. Perform hypothesis testing (ANOVA, Shapiro-Wilk) on property prices.
4. Calculate an advanced `undervalued_index` to find the best investment opportunities.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Preparation
Load the cleaned dataset and configure the target columns for our statistical tests.


In [ ]:
# Load the processed dataset
df = pd.read_csv("../data/processed/cleaned_data.csv")

# Define our core analytical columns
price_col = "price_egp"
area_col = "area_value"
location_col = "location_full"

# Ensure strict numeric types and drop invalid records
df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
df[area_col] = pd.to_numeric(df[area_col], errors='coerce')
df = df.dropna(subset=[price_col, area_col])

print(f"Data successfully loaded. Valid records for analysis: {len(df):,}")

## 2. Financial Metrics Generation
Calculate the fundamental real estate metric: **Price per Square Foot / Meter**.


In [ ]:
# Calculate normalized price
df['price_per_sqft'] = df[price_col] / df[area_col]

# Display summary statistics
print("========== DESCRIPTIVE STATISTICS ==========")
display(df[[price_col, area_col, 'price_per_sqft']].describe().round(2))

## 3. Correlation & Normality Tests
Evaluate how features correlate with price and check if prices follow a normal distribution using the **Shapiro-Wilk** test.


In [ ]:
print("========== CORRELATION MATRIX ==========")
display(df[[price_col, area_col, 'price_per_sqft']].corr().round(4))

# Shapiro-Wilk Normality Test (Sub-sampling for performance)
sample = df[price_col].dropna().sample(min(500, len(df)), random_state=42)
stat, p_val = stats.shapiro(sample)

print(f"\nShapiro-Wilk Test p-value: {p_val:.4e}")
if p_val < 0.05:
    print("Result: Prices are NOT normally distributed (Reject H0).")
else:
    print("Result: Prices ARE normally distributed (Fail to reject H0).")

## 4. Location Impact (ANOVA Testing)
Does location significantly impact the price per square foot? We use a **One-Way ANOVA** test across the top 5 most common locations to find out.


In [ ]:
top_locs = df[location_col].value_counts().head(5).index
groups = [df[df[location_col] == loc]['price_per_sqft'].dropna() for loc in top_locs]

f_stat, p_val = stats.f_oneway(*groups)

print("========== ANOVA TEST RESULTS ==========")
print(f"F-Statistic: {f_stat:.2f}")
print(f"P-Value: {p_val:.4e}")

if p_val < 0.05:
    print("Result: Location has a statistically SIGNIFICANT impact on property price.")
else:
    print("Result: Location does NOT have a significant impact.")

## 5. Undervaluation Analysis
Identify the best investment opportunities by comparing a property's `price_per_sqft` against the average for its specific location.

The `undervalued_index` represents the percentage by which a property is cheaper than its neighborhood average.


In [ ]:
# Calculate local averages
loc_avg = df.groupby(location_col)['price_per_sqft'].mean()
df['location_avg'] = df[location_col].map(loc_avg)

# Calculate Percentage Undervaluation
df['undervalued_index'] = (
    (df['location_avg'] - df['price_per_sqft']) / df['location_avg']
) * 100

# Extract top investments
undervalued = df.sort_values(by='undervalued_index', ascending=False)

print("========== TOP 10 UNDERVALUED PROPERTIES ==========")
display(undervalued[[location_col, 'price_per_sqft', 'location_avg', 'undervalued_index']].head(10).round(2))

## 6. Export Final Dataset
Save the enriched dataset containing the new metrics for ingestion into our Dashboard.


In [ ]:
# Save to processed directory
output_path = "../data/processed/final_dataset.csv"
df.to_csv(output_path, index=False)

print(f"✅ SUCCESS: Final dataset saved to {output_path}")